# Ch.2 — Dimensionality Reduction

> **Running theme:** The retail company's 6-feature customer spending data lives in a space we cannot visualise. PCA compresses it by keeping maximum variance. t-SNE reveals cluster topology. UMAP balances speed, global structure, and the ability to transform new customers. Each tool makes a different promise — know which promise was broken before trusting the picture.

## Core Idea

**PCA:** linear projection along eigenvectors of the covariance matrix, ordered by eigenvalue. Maximises retained variance. Invertible.

**t-SNE:** minimises KL divergence between high-d Gaussian and low-d Student-t similarity distributions. Preserves local neighbourhood topology. Distances between clusters are NOT meaningful.

**UMAP:** topological graph-matching. Faster than t-SNE, better global structure, has `transform()` for new customers. The only one of the three suitable as a downstream preprocessing step.

## Running Example

Dataset: **UCI Wholesale Customers** — 440 customers, 6 spending features (log-transformed + standardised).
All three methods project the 6-feature space down to 2D.
Colour: K-Means cluster labels from Ch.1 (used for visual validation, **not** for fitting).

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path

IMG = Path("img"); IMG.mkdir(exist_ok=True)

# ── Load and preprocess ───────────────────────────────────────────────────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(url)
spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicatessen']
X = df[spend_cols].values

X_log = np.log1p(X)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_log)

# K-Means labels from Ch.1 (for colouring only)
km5 = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42).fit(X_sc)
labels_km = km5.labels_

print(f"Dataset: {X.shape[0]} customers × {X.shape[1]} features")
print("Features:", spend_cols)

## PCA: Scree Plot and Explained Variance

In [ ]:
# ── PCA full scree ────────────────────────────────────────────────────────────
pca_full = PCA(n_components=X_sc.shape[1]).fit(X_sc)
evr = pca_full.explained_variance_ratio_
cumevr = evr.cumsum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(1, len(evr)+1), evr, color='steelblue', alpha=0.8)
ax1.set_xlabel('Principal Component'); ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Scree Plot — Individual EVR')
ax1.set_xticks(range(1, len(evr)+1))

ax2.plot(range(1, len(cumevr)+1), cumevr, 'r-o', markersize=8)
ax2.axhline(y=0.90, color='gray', linestyle='--', alpha=0.5, label='90% threshold')
ax2.set_xlabel('Number of Components'); ax2.set_ylabel('Cumulative EVR')
ax2.set_title('Cumulative Explained Variance')
ax2.set_xticks(range(1, len(cumevr)+1))
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(IMG / "ch02_scree_plot.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("Component breakdown:")
for i, (e, c) in enumerate(zip(evr, cumevr)):
    print(f"  PC{i+1}: {e:.1%} (cumulative: {c:.1%})")
k90 = (cumevr < 0.90).sum() + 1
print(f"\nComponents to reach 90% variance: {k90}")

## PCA: 2D Projection and Loadings

In [ ]:
# ── PCA 2D projection ─────────────────────────────────────────────────────────
pca2 = PCA(n_components=2, random_state=42)
X_pca = pca2.fit_transform(X_sc)
print(f"PCA 2D retains {pca2.explained_variance_ratio_.sum()*100:.1f}% of variance")

# Loadings — what do the components mean?
loadings = pd.DataFrame(pca2.components_.T, index=spend_cols, columns=['PC1', 'PC2'])
print("\nPCA Loadings (feature contributions to each component):")
print(loadings.round(3))
print("\nInterpretation:")
print("  PC1: positive = high overall spend → 'Total Spend Magnitude'")
print("  PC2: separates Fresh/Frozen (positive) from Grocery/Detergents (negative) → 'Product Mix'")

In [ ]:
# ── PCA 2D scatter, coloured by K-Means labels ────────────────────────────────
segment_names = ["Loyalists", "Price-Sensitive", "Big Spenders",
                 "Occasional Buyers", "Deli Specialists"]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(5):
    mask = labels_km == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[i],
                    s=20, alpha=0.6, label=segment_names[i])
axes[0].set_title('PCA 2D — K-Means Segments')
axes[0].set_xlabel('PC1 (Total Spend)'); axes[0].set_ylabel('PC2 (Product Mix)')
axes[0].legend(fontsize=8, markerscale=2)

# Loadings biplot overlay
for i, feat in enumerate(spend_cols):
    axes[1].arrow(0, 0, loadings.iloc[i, 0]*3, loadings.iloc[i, 1]*3,
                  head_width=0.08, head_length=0.05, fc='red', ec='red', alpha=0.7)
    axes[1].text(loadings.iloc[i, 0]*3.3, loadings.iloc[i, 1]*3.3, feat, fontsize=8, color='red')
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c='gray', s=5, alpha=0.2)
axes[1].set_title('PCA Biplot — Feature Loadings')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)

plt.tight_layout()
fig.savefig(IMG / "ch02_pca_2d.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## t-SNE: Perplexity Sweep

t-SNE preserves **local** neighbourhood structure. The `perplexity` parameter controls the effective number of neighbours. We try 10, 30, and 50 to see how cluster appearance changes.

⚠️ **Distances between clusters in t-SNE are NOT meaningful** — only topology is.

In [ ]:
# ── t-SNE perplexity sweep ────────────────────────────────────────────────────
perplexities = [10, 30, 50]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, learning_rate='auto',
                init='pca', random_state=42)
    X_tsne = tsne.fit_transform(X_sc)
    for i in range(5):
        mask = labels_km == i
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=colors[i],
                   s=15, alpha=0.6, label=segment_names[i])
    ax.set_title(f't-SNE (perplexity={perp})')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')

axes[0].legend(fontsize=7, markerscale=2)
plt.suptitle('t-SNE: perplexity controls local vs global emphasis', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch02_tsne_perplexity.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## UMAP: n_neighbors Sweep

UMAP preserves **both local and global** structure. `n_neighbors` controls the balance.
Unlike t-SNE, UMAP has `transform()` for new data.

In [ ]:
# ── UMAP n_neighbors sweep ────────────────────────────────────────────────────
try:
    import umap

    n_neighbors_list = [5, 15, 50]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, nn in zip(axes, n_neighbors_list):
        reducer = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1, random_state=42)
        X_umap = reducer.fit_transform(X_sc)
        for i in range(5):
            mask = labels_km == i
            ax.scatter(X_umap[mask, 0], X_umap[mask, 1], c=colors[i],
                       s=15, alpha=0.6, label=segment_names[i])
        ax.set_title(f'UMAP (n_neighbors={nn})')
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

    axes[0].legend(fontsize=7, markerscale=2)
    plt.suptitle('UMAP: n_neighbors controls local vs global structure', y=1.02)
    plt.tight_layout()
    fig.savefig(IMG / "ch02_umap_neighbors.png", dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

except ImportError:
    print("umap-learn not installed — run: pip install umap-learn")

## PCA vs t-SNE vs UMAP: Side-by-Side Comparison

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
X_pca2 = PCA(n_components=2, random_state=42).fit_transform(X_sc)
X_tsne2 = TSNE(n_components=2, perplexity=30, learning_rate='auto',
               init='pca', random_state=42).fit_transform(X_sc)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['PCA', 't-SNE (perplexity=30)', 'UMAP (n_neighbors=15)']

try:
    import umap
    X_umap2 = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                         random_state=42).fit_transform(X_sc)
    embeddings = [X_pca2, X_tsne2, X_umap2]
except ImportError:
    embeddings = [X_pca2, X_tsne2, X_pca2]
    titles[2] = 'UMAP (not installed)'

for ax, Xemb, title in zip(axes, embeddings, titles):
    for i in range(5):
        mask = labels_km == i
        ax.scatter(Xemb[mask, 0], Xemb[mask, 1], c=colors[i],
                   s=15, alpha=0.6, label=segment_names[i])
    ax.set_title(title)

axes[0].legend(fontsize=7, markerscale=2)
plt.suptitle('Three Projections of 440 Customers — Same Clusters, Different Views', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch02_comparison.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## Re-Clustering in PCA Space: Does Dimensionality Reduction Help?

Hypothesis: 6D distances are noisy (curse of dimensionality). Clustering in PCA 2D should give tighter segments.

In [ ]:
# ── Re-cluster in PCA 2D ──────────────────────────────────────────────────────
km_6d = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42).fit(X_sc)
sil_6d = silhouette_score(X_sc, km_6d.labels_)

km_pca = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42).fit(X_pca2)
sil_pca = silhouette_score(X_pca2, km_pca.labels_)

print(f"Silhouette in 6D (raw):   {sil_6d:.4f}")
print(f"Silhouette in PCA 2D:     {sil_pca:.4f}")
print(f"Improvement:              +{sil_pca - sil_6d:.4f}")
print(f"\n{'✅ PCA helps!' if sil_pca > sil_6d else '⚠️ PCA did not help'}")
print(f"Still {'below' if sil_pca < 0.5 else 'above'} the 0.5 target — Ch.3 will push further.")

## What Can Go Wrong: t-SNE Distance Lie

Distances between clusters in t-SNE are meaningless. This demo shows two groups that are VERY different in original space but appear close in t-SNE, and vice versa.

In [ ]:
# ── t-SNE distance warning ────────────────────────────────────────────────────
# Compute actual centroid distances in 6D scaled space
centroids_6d = km_6d.cluster_centers_
from scipy.spatial.distance import pdist, squareform
dist_6d = squareform(pdist(centroids_6d, metric='euclidean'))

# Compute centroid positions in t-SNE space
centroids_tsne = np.array([X_tsne2[labels_km == i].mean(axis=0) for i in range(5)])
dist_tsne = squareform(pdist(centroids_tsne, metric='euclidean'))

print("Centroid distances in 6D (true distances):")
print(pd.DataFrame(dist_6d.round(2), index=segment_names, columns=segment_names))
print("\nCentroid distances in t-SNE (DO NOT TRUST):")
print(pd.DataFrame(dist_tsne.round(1), index=segment_names, columns=segment_names))
print("\n⚠️ t-SNE distances are not proportional to true distances!")

## What Can Go Wrong: PCA Misses Non-Linear Structure

In [ ]:
# ── PCA reconstruction error ──────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error

for n_comp in [2, 3, 4, 5]:
    pca_k = PCA(n_components=n_comp, random_state=42)
    X_reduced = pca_k.fit_transform(X_sc)
    X_reconstructed = pca_k.inverse_transform(X_reduced)
    mse = mean_squared_error(X_sc, X_reconstructed)
    evr_total = pca_k.explained_variance_ratio_.sum()
    print(f"  PCA({n_comp}): EVR={evr_total:.1%}, reconstruction MSE={mse:.4f}")

print("\n2 PCs lose 28% of variance — if cluster separation depends on")
print("the lost dimensions, silhouette suffers. Try 4 PCs as a middle ground.")

## Exercises

1. **UMAP min_dist sweep.** Try `min_dist` ∈ {0.0, 0.1, 0.5, 1.0} with `n_neighbors=15`. Plot all four embeddings. How does min_dist affect cluster compactness?

2. **PCA as preprocessing for K-Means.** Run K-Means (K=5) on PCA with 2, 3, 4, and 5 components. Compute silhouette for each. Which number of components gives the best clustering?

3. **t-SNE reproducibility test.** Run t-SNE (perplexity=30) with 5 different `random_state` values. Plot all 5 embeddings. How much does the layout change? Does cluster membership change?

In [ ]:
# Exercise 1 — UMAP min_dist sweep
# TODO: your solution here
pass

In [ ]:
# Exercise 2 — PCA preprocessing for K-Means
# TODO: your solution here
pass

In [ ]:
# Exercise 3 — t-SNE reproducibility
# TODO: your solution here
pass